In [ ]:
# 1. 라이브러리 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use("seaborn-v0_8")

In [ ]:
# 2. 종목 티커 설정

# KOSPI 대표주
kospi_tickers = [
    "005930.KS",  # Samsung Electronics
    "000660.KS",  # SK Hynix
    "207940.KS",  # Samsung Biologics
    "035420.KS",  # NAVER
    "035720.KS",  # Kakao
    "051910.KS",  # LG Chem
    "068270.KS",  # Celltrion
    "105560.KS",  # KB Financial
    "055550.KS",  # Shinhan Financial
    "006400.KS",  # Samsung SDI
    "012330.KS",  # Hyundai Mobis
    "000270.KS",  # Kia
    "096770.KS",  # SK Innovation
    "034730.KS",  # SK
    "017670.KS",  # SK Telecom
]

# Nikkei 대표주 (일본)
nikkei_tickers = [
    "7203.T",  # Toyota
    "6758.T",  # Sony
    "9432.T",  # NTT
    "9984.T",  # SoftBank Group
    "8306.T",  # Mitsubishi UFJ
    "7267.T",  # Honda
    "8035.T",  # Tokyo Electron
    "6954.T",  # Fanuc
    "7733.T",  # Olympus
    "6098.T",  # Recruit Holdings
    "4063.T",  # Shin-Etsu Chemical
    "6861.T",  # Keyence
    "6367.T",  # Daikin
    "8031.T",  # Mitsui
    "8316.T",  # Sumitomo Mitsui Financial
]

# 항셍지수 대표주 (홍콩)
hsi_tickers = [
    "0700.HK",  # Tencent
    "0939.HK",  # CCB
    "1299.HK",  # AIA
    "0005.HK",  # HSBC
    "1398.HK",  # ICBC
    "0941.HK",  # China Mobile
    "0388.HK",  # HKEX
    "2318.HK",  # Ping An
    "3988.HK",  # Bank of China
    "2628.HK",  # China Life
    "0001.HK",  # CK Hutchison
    "0002.HK",  # CLP
    "0011.HK",  # Hang Seng Bank
    "1928.HK",  # Sands China
    "0883.HK",  # CNOOC
]

tickers = kospi_tickers + nikkei_tickers + hsi_tickers
start_date = "2017-01-01"
end_date = "2025-12-31"

In [ ]:
# 3. 데이터 다운로드 및 월별 수익률 계산

def download_price_data(tickers, start="2017-01-01", end="2025-12-31"):
    """Download adjusted close prices for given tickers using yfinance."""
    data = yf.download(
        tickers=" ".join(tickers),
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
    )
    if isinstance(data, pd.DataFrame) and "Adj Close" in data.columns:
        prices = data["Adj Close"]
    else:
        prices = data
    prices = prices.dropna(how="all")
    return prices


def compute_monthly_data(prices: pd.DataFrame):
    """Convert daily prices to month-end prices and compute returns."""
    # 최신 pandas에서는 'M' 대신 'ME'(month-end) 사용
    monthly_prices = prices.resample("ME").last()
    monthly_returns = monthly_prices.pct_change()
    return monthly_prices, monthly_returns


print("Downloading price data from yfinance...")
prices = download_price_data(tickers, start=start_date, end=end_date)
print("Computing monthly prices and returns...")
monthly_prices, monthly_returns = compute_monthly_data(prices)
monthly_returns.head()

In [ ]:
# 4. 모멘텀(최근 12개월 수익률) 계산

def compute_momentum_scores(monthly_returns: pd.DataFrame, lookback_months: int = 12):
    """Compute 12-month momentum scores from monthly returns.

    We use compounded returns over the past `lookback_months`, shifted by 1 month
    to avoid look-ahead bias.
    """
    compounded = (1 + monthly_returns).rolling(lookback_months).apply(
        np.prod, raw=True
    ) - 1
    momentum = compounded.shift(1)
    return momentum


print("Computing 12-month momentum scores...")
momentum_scores = compute_momentum_scores(monthly_returns, lookback_months=12)
momentum_scores.tail()

In [ ]:
# 5. 백테스트 실행 (매월 모멘텀 상위 10개 동일비중 투자)

def backtest_momentum_strategy(
    monthly_returns: pd.DataFrame,
    momentum_scores: pd.DataFrame,
    start_year: int = 2018,
    end_year: int = 2025,
    top_n: int = 10,
):
    """Run a monthly rebalanced momentum strategy.

    At each month-end, pick top_n tickers by 12-month momentum and hold them
    equally-weighted for the next month.
    """
    portfolio_rets = []
    dates = []

    for i in range(len(momentum_scores.index) - 1):
        date = momentum_scores.index[i]
        next_date = momentum_scores.index[i + 1]

        if not (start_year <= date.year <= end_year):
            continue

        scores = momentum_scores.iloc[i].dropna()
        if scores.empty:
            continue

        top_tickers = scores.sort_values(ascending=False).head(top_n).index

        next_month_rets = monthly_returns.loc[next_date, top_tickers].dropna()
        if next_month_rets.empty:
            continue

        weight = 1.0 / len(next_month_rets)
        portfolio_ret = (next_month_rets * weight).sum()
        portfolio_rets.append(portfolio_ret)
        dates.append(next_date)

    portfolio_rets = pd.Series(portfolio_rets, index=pd.DatetimeIndex(dates))
    return portfolio_rets


print("Running momentum strategy backtest...")
portfolio_returns = backtest_momentum_strategy(
    monthly_returns=monthly_returns,
    momentum_scores=momentum_scores,
    start_year=2018,
    end_year=2025,
    top_n=10,
)
portfolio_returns.head()

In [ ]:
# 6. 결과 시각화 (누적 수익률 그래프)

def plot_cumulative_returns(cum_returns: pd.Series):
    plt.figure(figsize=(10, 6))
    plt.plot(cum_returns.index, cum_returns.values, label="Asia Momentum Strategy")
    plt.title("Asia Momentum Strategy Cumulative Returns (2018-2025)")
    plt.xlabel("Date")
    plt.ylabel("Cumulative Return (Growth of 1)")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


cum_returns = (1 + portfolio_returns).cumprod()
plot_cumulative_returns(cum_returns)

In [ ]:
# 7. Sharpe ratio, MDD 계산 및 출력

def compute_performance_metrics(portfolio_returns: pd.Series):
    """Compute cumulative returns, Sharpe ratio (annualized), and MDD."""
    cum_returns = (1 + portfolio_returns).cumprod()

    # Annualized Sharpe ratio (monthly data, risk-free rate = 0)
    mean_ret = portfolio_returns.mean()
    std_ret = portfolio_returns.std()
    sharpe = np.nan
    if std_ret != 0 and not np.isnan(std_ret):
        sharpe = (mean_ret / std_ret) * np.sqrt(12)

    # Maximum Drawdown
    running_max = cum_returns.cummax()
    drawdowns = cum_returns / running_max - 1.0
    mdd = drawdowns.min()

    return cum_returns, sharpe, mdd


cum_returns, sharpe, mdd = compute_performance_metrics(portfolio_returns)
print("Backtest Results (2018-2025)")
print("-------------------------------------")
print(f"Final cumulative return: {cum_returns.iloc[-1]:.2f}x")
print(f"Annualized Sharpe ratio: {sharpe:.3f}")
print(f"Maximum drawdown (MDD): {mdd:.2%}")